# Lab Assignment 2: Data Cleaning, Integration, and Exploration
**PGDDS 103: Data Analysis & Data Visualizations**

**Case:** FreshCart Grocers — the VP of Customer Success is alarmed by recent subscription cancellations.

**Data files**
1. `freshcart_users.csv` — shopper demographics, plan tier, tenure, total orders, churn status
2. `freshcart_refunds.csv` — logs of orders where items were reported missing or damaged

**Tasks covered in this notebook**
1. Data Cleaning — drop duplicate rows, median-impute `Age`
2. Anomaly Correction — negative `Total_Orders` → 0; `Months_Active == 0` while `Total_Orders > 0` → 1
3. Variable Creation — `Monthly_Order_Rate`, `Tenure_Cohort`
4. Data Integration — left join the refunds log on `ShopperID`, fill `Missing_Items_Reported` NaN with 0
5. Exploration — group by `Churn`; cross-tabulate `Tenure_Cohort` vs `Churn`
6. Validation — `assert` zero missing values and zero duplicate `ShopperID` entries

## 0. Setup

Run this cell first. It imports the libraries and locates the two CSV files.

**How the file loading works in Google Colab**
- If you already uploaded `freshcart_users.csv` and `freshcart_refunds.csv` to Colab, they sit in `/content/` and are picked up automatically.
- If they are not found, the notebook writes the assignment's sample data to disk so the notebook still runs end-to-end.
- To upload manually instead, uncomment the `files.upload()` block below.

In [1]:
import os
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 50)

print("pandas:", pd.__version__)
print("numpy :", np.__version__)

# --- Optional: manual upload in Google Colab -------------------------------
# from google.colab import files
# uploaded = files.upload()   # then pick freshcart_users.csv and freshcart_refunds.csv
# ---------------------------------------------------------------------------

pandas: 3.0.5
numpy : 2.4.6


In [2]:
# Fallback copies of the assignment data, used only if the CSVs are not found.
USERS_FALLBACK = """ShopperID,Age,Plan_Tier,Months_Active,Total_Orders,Churn
F-301,38,Family,18,72,0
F-302,,Solo,2,3,1
F-303,50,Family,26,104,0
F-301,38,Family,18,72,0
F-304,27,Solo,6,-1,1
F-305,33,Duo,0,2,0
F-306,45,Family,14,56,0
F-307,,Solo,3,4,1
"""

REFUNDS_FALLBACK = """ShopperID,Missing_Items_Reported
F-302,4
F-304,6
F-307,3
"""


def locate(filename, fallback_text, extra_names=()):
    """Return a readable path for `filename`, writing the fallback data if needed."""
    candidates = []
    for name in (filename,) + tuple(extra_names):
        candidates += [name, f"/content/{name}", f"data/{name}", f"/content/data/{name}"]
    for path in candidates:
        if os.path.exists(path):
            print(f"Found: {path}")
            return path
    with open(filename, "w") as fh:
        fh.write(fallback_text)
    print(f"Not found on disk -> wrote sample data to: {filename}")
    return filename


users_path = locate("freshcart_users.csv", USERS_FALLBACK, extra_names=("freshcart_users1.csv",))
refunds_path = locate("freshcart_refunds.csv", REFUNDS_FALLBACK, extra_names=("freshcart_refunds1.csv",))

Found: data/freshcart_users.csv
Found: data/freshcart_refunds.csv


## 1. Load the raw data and inspect it

In [3]:
users_raw = pd.read_csv(users_path)
refunds_raw = pd.read_csv(refunds_path)

print("users shape  :", users_raw.shape)
print("refunds shape:", refunds_raw.shape)
users_raw

users shape  : (8, 6)
refunds shape: (3, 2)


,ShopperID,Age,Plan_Tier,Months_Active,Total_Orders,Churn
0,F-301,38.0,Family,18,72,0
1,F-302,NaN,Solo,2,3,1
2,F-303,50.0,Family,26,104,0
3,F-301,38.0,Family,18,72,0
4,F-304,27.0,Solo,6,-1,1
5,F-305,33.0,Duo,0,2,0
6,F-306,45.0,Family,14,56,0
7,F-307,NaN,Solo,3,4,1


In [4]:
refunds_raw

,ShopperID,Missing_Items_Reported
0,F-302,4
1,F-304,6
2,F-307,3


In [5]:
print("--- dtypes ---")
print(users_raw.dtypes)
print("\n--- missing values per column ---")
print(users_raw.isna().sum())
print("\n--- duplicate rows:", int(users_raw.duplicated().sum()), "---")
print("\n--- summary ---")
users_raw.describe(include="all")

--- dtypes ---
ShopperID            str
Age              float64
Plan_Tier            str
Months_Active      int64
Total_Orders       int64
Churn              int64
dtype: object

--- missing values per column ---
ShopperID        0
Age              2
Plan_Tier        0
Months_Active    0
Total_Orders     0
Churn            0
dtype: int64

--- duplicate rows: 1 ---

--- summary ---


,ShopperID,Age,Plan_Tier,Months_Active,Total_Orders,Churn
count,8,6.000000,8,8.000000,8.000000,8.000000
unique,7,NaN,3,NaN,NaN,NaN
top,F-301,NaN,Family,NaN,NaN,NaN
freq,2,NaN,4,NaN,NaN,NaN
mean,NaN,38.500000,NaN,10.875000,39.000000,0.375000
std,NaN,8.215838,NaN,9.433035,41.716732,0.517549
min,NaN,27.000000,NaN,0.000000,-1.000000,0.000000
25%,NaN,34.250000,NaN,2.750000,2.750000,0.000000
50%,NaN,38.000000,NaN,10.000000,30.000000,0.000000
75%,NaN,43.250000,NaN,18.000000,72.000000,1.000000


## 2. Data Cleaning

- Remove duplicate rows (`F-301` appears twice).
- Impute the missing `Age` values with the **median** age (median is robust to outliers, unlike the mean).

In [6]:
df = users_raw.copy()

before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Duplicate rows removed: {before - len(df)}  ({before} -> {len(df)} rows)")

median_age = df["Age"].median()
n_missing_age = int(df["Age"].isna().sum())
df["Age"] = df["Age"].fillna(median_age)
print(f"Median Age = {median_age}  |  Age values imputed: {n_missing_age}")

df

Duplicate rows removed: 1  (8 -> 7 rows)
Median Age = 38.0  |  Age values imputed: 2


,ShopperID,Age,Plan_Tier,Months_Active,Total_Orders,Churn
0,F-301,38.0,Family,18,72,0
1,F-302,38.0,Solo,2,3,1
2,F-303,50.0,Family,26,104,0
3,F-304,27.0,Solo,6,-1,1
4,F-305,33.0,Duo,0,2,0
5,F-306,45.0,Family,14,56,0
6,F-307,38.0,Solo,3,4,1


## 3. Anomaly Correction

- `Total_Orders` cannot be negative → clip any negative value to **0**.
- A shopper cannot have orders with zero tenure → where `Months_Active == 0` **and** `Total_Orders > 0`, set `Months_Active = 1`.
  (This also keeps `Monthly_Order_Rate` from dividing by zero in the next step.)

In [7]:
neg_mask = df["Total_Orders"] < 0
print("Rows with negative Total_Orders:", int(neg_mask.sum()))
print(df.loc[neg_mask, ["ShopperID", "Total_Orders"]].to_string(index=False) if neg_mask.any() else "(none)")
df.loc[neg_mask, "Total_Orders"] = 0

tenure_mask = (df["Months_Active"] == 0) & (df["Total_Orders"] > 0)
print("\nRows with Months_Active == 0 but Total_Orders > 0:", int(tenure_mask.sum()))
print(df.loc[tenure_mask, ["ShopperID", "Months_Active", "Total_Orders"]].to_string(index=False)
      if tenure_mask.any() else "(none)")
df.loc[tenure_mask, "Months_Active"] = 1

df

Rows with negative Total_Orders: 1
ShopperID  Total_Orders
    F-304            -1

Rows with Months_Active == 0 but Total_Orders > 0: 1
ShopperID  Months_Active  Total_Orders
    F-305              0             2


,ShopperID,Age,Plan_Tier,Months_Active,Total_Orders,Churn
0,F-301,38.0,Family,18,72,0
1,F-302,38.0,Solo,2,3,1
2,F-303,50.0,Family,26,104,0
3,F-304,27.0,Solo,6,0,1
4,F-305,33.0,Duo,1,2,0
5,F-306,45.0,Family,14,56,0
6,F-307,38.0,Solo,3,4,1


## 4. Variable Creation

- `Monthly_Order_Rate = Total_Orders / Months_Active` — orders per active month, which makes shoppers with different tenures comparable.
- `Tenure_Cohort` — a categorical bucket: **0-6 Mos**, **7-12 Mos**, **13+ Mos**.

In [8]:
df["Monthly_Order_Rate"] = (df["Total_Orders"] / df["Months_Active"]).round(2)

bins = [0, 6, 12, np.inf]
labels = ["0-6 Mos", "7-12 Mos", "13+ Mos"]
df["Tenure_Cohort"] = pd.cut(df["Months_Active"], bins=bins, labels=labels, right=True)

df[["ShopperID", "Months_Active", "Total_Orders", "Monthly_Order_Rate", "Tenure_Cohort", "Churn"]]

,ShopperID,Months_Active,Total_Orders,Monthly_Order_Rate,Tenure_Cohort,Churn
0,F-301,18,72,4.00,13+ Mos,0
1,F-302,2,3,1.50,0-6 Mos,1
2,F-303,26,104,4.00,13+ Mos,0
3,F-304,6,0,0.00,0-6 Mos,1
4,F-305,1,2,2.00,0-6 Mos,0
5,F-306,14,56,4.00,13+ Mos,0
6,F-307,3,4,1.33,0-6 Mos,1


## 5. Data Integration

Left join `freshcart_refunds.csv` on `ShopperID` so every shopper is kept, then fill the
`Missing_Items_Reported` NaNs with **0** — no refund log simply means no missing items were reported.

In [9]:
merged = df.merge(refunds_raw, on="ShopperID", how="left")
print("Missing_Items_Reported NaNs after the left join:", int(merged["Missing_Items_Reported"].isna().sum()))

merged["Missing_Items_Reported"] = merged["Missing_Items_Reported"].fillna(0).astype(int)
print("Missing_Items_Reported NaNs after imputing with 0:", int(merged["Missing_Items_Reported"].isna().sum()))

merged

Missing_Items_Reported NaNs after the left join: 4
Missing_Items_Reported NaNs after imputing with 0: 0


,ShopperID,Age,Plan_Tier,Months_Active,Total_Orders,Churn,Monthly_Order_Rate,Tenure_Cohort,Missing_Items_Reported
0,F-301,38.0,Family,18,72,0,4.00,13+ Mos,0
1,F-302,38.0,Solo,2,3,1,1.50,0-6 Mos,4
2,F-303,50.0,Family,26,104,0,4.00,13+ Mos,0
3,F-304,27.0,Solo,6,0,1,0.00,0-6 Mos,6
4,F-305,33.0,Duo,1,2,0,2.00,0-6 Mos,0
5,F-306,45.0,Family,14,56,0,4.00,13+ Mos,0
6,F-307,38.0,Solo,3,4,1,1.33,0-6 Mos,3


## 6. Exploration

### 6a. Average `Monthly_Order_Rate` and `Missing_Items_Reported` by churn status

In [10]:
churn_summary = (
    merged.groupby("Churn")[["Monthly_Order_Rate", "Missing_Items_Reported"]]
    .mean()
    .round(2)
    .rename(index={0: "0 (Retained)", 1: "1 (Churned)"})
)
churn_summary

,Monthly_Order_Rate,Missing_Items_Reported
Churn,,
0 (Retained),3.50,0.00
1 (Churned),0.94,4.33


In [11]:
# Same view with shopper counts alongside the averages.
churn_detail = (
    merged.groupby("Churn")
    .agg(
        Shoppers=("ShopperID", "count"),
        Avg_Monthly_Order_Rate=("Monthly_Order_Rate", "mean"),
        Avg_Missing_Items_Reported=("Missing_Items_Reported", "mean"),
    )
    .round(2)
)
churn_detail

,Shoppers,Avg_Monthly_Order_Rate,Avg_Missing_Items_Reported
Churn,,,
0,4,3.50,0.00
1,3,0.94,4.33


### 6b. Cross-tabulation of `Tenure_Cohort` vs `Churn`

In [12]:
cohort_churn = pd.crosstab(merged["Tenure_Cohort"], merged["Churn"], margins=True, margins_name="Total")
cohort_churn

Churn,0,1,Total
Tenure_Cohort,,,
0-6 Mos,1,3,4
13+ Mos,3,0,3
Total,4,3,7


In [13]:
# Row percentages: churn rate within each tenure cohort.
cohort_rate = (
    pd.crosstab(merged["Tenure_Cohort"], merged["Churn"], normalize="index")
    .mul(100)
    .round(1)
    .rename(columns={0: "% Retained", 1: "% Churned"})
)
cohort_rate

Churn,% Retained,% Churned
Tenure_Cohort,,
0-6 Mos,25.0,75.0
13+ Mos,100.0,0.0


## 7. Validation

`assert` statements fail loudly if the cleaning did not hold, so the checks below are the proof that the
final dataset is fit for analysis.

In [14]:
# 1. No missing values anywhere in the final dataset.
assert merged.isna().sum().sum() == 0, f"Missing values remain:\n{merged.isna().sum()}"

# 2. No duplicate ShopperID entries.
assert merged["ShopperID"].duplicated().sum() == 0, "Duplicate ShopperID entries remain"

# 3. Supporting checks for the anomaly corrections.
assert (merged["Total_Orders"] >= 0).all(), "Negative Total_Orders remain"
assert not ((merged["Months_Active"] == 0) & (merged["Total_Orders"] > 0)).any(), "Tenure anomaly remains"

print("All assertions passed.")
print("Missing values total     :", int(merged.isna().sum().sum()))
print("Duplicate ShopperID count:", int(merged["ShopperID"].duplicated().sum()))
print("Final dataset shape      :", merged.shape)

All assertions passed.
Missing values total     : 0
Duplicate ShopperID count: 0
Final dataset shape      : (7, 9)


## 8. Final cleaned dataset and findings

In [15]:
merged

,ShopperID,Age,Plan_Tier,Months_Active,Total_Orders,Churn,Monthly_Order_Rate,Tenure_Cohort,Missing_Items_Reported
0,F-301,38.0,Family,18,72,0,4.00,13+ Mos,0
1,F-302,38.0,Solo,2,3,1,1.50,0-6 Mos,4
2,F-303,50.0,Family,26,104,0,4.00,13+ Mos,0
3,F-304,27.0,Solo,6,0,1,0.00,0-6 Mos,6
4,F-305,33.0,Duo,1,2,0,2.00,0-6 Mos,0
5,F-306,45.0,Family,14,56,0,4.00,13+ Mos,0
6,F-307,38.0,Solo,3,4,1,1.33,0-6 Mos,3


In [16]:
merged.to_csv("freshcart_clean.csv", index=False)
print("Saved -> freshcart_clean.csv")

# In Colab, uncomment to download the cleaned file:
# from google.colab import files
# files.download("freshcart_clean.csv")

Saved -> freshcart_clean.csv


### Findings for the VP of Customer Success

Read these against the tables above (the sample is small, so treat them as directional):

1. **Churned shoppers order far less often.** Their average `Monthly_Order_Rate` is a fraction of the
   retained group's — low engagement shows up in the order rate well before the cancellation does.
2. **Churn tracks with missing/damaged items.** Every shopper in the refunds log churned, and the average
   `Missing_Items_Reported` for churned shoppers is markedly higher than for retained ones. Fulfilment
   quality looks like the lever, not price.
3. **Churn is concentrated in the 0-6 month cohort.** All three churners sit in the newest tenure bucket
   (75% of that cohort), while the 13+ month cohort is 100% retained and no shopper falls in the 7-12 month
   bucket — the risk window is early.

**Recommended actions:** monitor `Monthly_Order_Rate` as an early-warning signal in the first six months,
and treat any missing-items report from a new subscriber as a retention trigger (proactive credit + follow-up).